# 04 — Model Training (Stage 2)
Run hyperopt + final training + export in a notebook for interactive exploration.
For production / reproducible runs use `scripts/run_training.py` instead.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt

from wdm.config import load_config
from wdm.model.dataset import build_dataset
from wdm.model.feature_pruner import maybe_prune_to_final
from wdm.model.tuner import run_hyperopt
from wdm.model.trainer import train_final
from wdm.model.evaluator import evaluate_all

PRODUCT = 'hzz_day'                               # 任何 configs/products/<name>.yaml
VERSION = 'v1_auto'                               # selected_features 版本
RUN_ID  = 'nb04_interactive'                      # 漏斗审计产物会落到此 run dir

cfg = load_config(PRODUCT)
data = build_dataset(cfg, version=VERSION)
print('train/valid/oot shapes:', data.X_train.shape, data.X_valid.shape, data.X_oot.shape)
print('base features (pre-funnel):', len(data.base_feature_list))

## 1.5 Stage-2 候选 → 最终漏斗

`scripts/run_training.py` 在 `build_dataset` 之后会调一次 `maybe_prune_to_final`，把候选池砍到 `final_feature_count`。这里复刻同一动作让交互式训练跟生产路径一致；产物 `exploratory_importance.csv` / `pruned_features.txt` 由 `notebooks/07_stage2_funnel_audit.ipynb` 消费。

若 `stage2_candidate_count` 没设或 ≤ `final_feature_count`，本段是 no-op，不会写文件。

In [ ]:
run_dir = Path(cfg['_repo_root']) / 'artifacts' / PRODUCT / 'models' / RUN_ID
run_dir.mkdir(parents=True, exist_ok=True)

n_before = len(data.base_feature_list)
data = maybe_prune_to_final(data, cfg, run_dir)
n_after = len(data.base_feature_list)

tcfg = cfg['training']
candidate_n = tcfg.get('stage2_candidate_count')
final_n = int(tcfg['final_feature_count'])
ranker = (tcfg.get('stage2_pruning') or {}).get('ranking_method', 'stability')
print(f'stage2_candidate_count : {candidate_n}')
print(f'final_feature_count    : {final_n}')
print(f'ranking_method         : {ranker}')
print(f'base features {n_before} → {n_after} (dropped {n_before - n_after})')

imp_csv = run_dir / 'exploratory_importance.csv'
if imp_csv.is_file():
    imp = pd.read_csv(imp_csv).sort_values('score', ascending=False).reset_index(drop=True)
    print(f'\nexploratory_importance.csv top-20 (ranker={imp["ranking_method"].iloc[0]}):')
    print(imp.head(20)[['feature', 'score', 'kept']].to_string(index=False))
    print(f'\n切线 ±5：')
    cut = int(imp["kept"].sum())
    print(imp.iloc[max(0, cut - 5): cut + 5][['feature', 'score', 'kept']].to_string(index=False))
else:
    print('\n（漏斗未触发——见 notebooks/07_stage2_funnel_audit.ipynb 的解释）')

In [ ]:
best_params, best_loss, trials = run_hyperopt(data.X_train, data.y_train, cfg, max_evals=10)
print('best params:', best_params)
print('best CV PR-AUC:', -best_loss)

In [ ]:
# Plot hyperopt trial loss over time
losses = [t['result']['loss'] for t in trials.trials]
plt.plot(range(1, len(losses) + 1), [-l for l in losses], marker='o')
plt.xlabel('trial')
plt.ylabel('CV PR-AUC')
plt.title('Hyperopt trajectory')
plt.grid(alpha=0.3)

In [ ]:
booster, evals = train_final(best_params, data.X_train, data.y_train, data.X_valid, data.y_valid, cfg)
metrics_df, binned, scores, imp = evaluate_all(booster, data, cfg)
metrics_df